In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
from IPython.display import Image
from google.colab import userdata
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import Runnable
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pathlib import Path
from pydantic import SecretStr
from typing import TypedDict

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def display_graph(runnable: Runnable, output_png: Path) -> None:
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

In [ ]:
class EditorialState(TypedDict):
    topic: str
    research: str
    draft: str
    translated_bg: str
    translated_it: str

In [ ]:
RESEARCHER_PROMPT = """You are the research desk in a small editorial team.
Return 2 to 3 crisp, concrete facts about the topic.
Do not repeat facts that already exist.
Keep each fact to one sentence.
"""

WRITER_PROMPT = """You are the writer in a small editorial team.
Write one short, readable paragraph that uses the provided facts.
Make it specific, simple, and accurate.
"""

TRANSLATOR_PROMPT = PromptTemplate.from_template("""You are the tranlation desk in a small editorial team.
You will be given a text in English and your main task is to translate it in {language}.
""")

web_search_tool = {"type": "web_search"}
researcher_model = ChatOpenAI(model="gpt-5-mini", api_key=openai_api_key, reasoning_effort="low").bind_tools([web_search_tool])

writer_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low")

translator_model = ChatOpenAI(model="gpt-5.4-mini", api_key=openai_api_key, reasoning_effort="low")

In [ ]:
def research(state: EditorialState):
    topic = state.get('topic')

    print("Initiating \"research\"...")

    response = researcher_model.invoke([
        SystemMessage(RESEARCHER_PROMPT),
        HumanMessage(f"Topic you need to research: {topic}")
    ])

    print("Research result:")

    research = "\n".join(chunk['text'] for chunk in response.content if chunk['type'] == "text")
    print("Extracted:")
    print(research)
    print()

    return { 'research': research }


def write(state: EditorialState):
    research = state.get('research')

    print("Initiating \"write\"...")

    result = writer_model.invoke([
        SystemMessage(WRITER_PROMPT),
        HumanMessage(f"Current research:\n{research}")
    ])

    draft = result.content
    print("Draft:")
    print(draft)

    return { 'draft': draft }

def create_translate_node(language: str, result_key: str):
    def translate(state: EditorialState):
        draft = state.get('draft')

        print("Initiating \"translate\"...")

        result = translator_model.invoke([
            SystemMessage(TRANSLATOR_PROMPT.format(language=language)),
            HumanMessage(f"Translate:\n{draft}")
        ])

        translated = result.content

        print("Translated:")
        print(translated)

        return { result_key: translated }

    return translate

In [ ]:
graph_builder = StateGraph(EditorialState)

graph_builder.add_node("research", research)
graph_builder.add_node("write", write)
graph_builder.add_node("translate_bg", create_translate_node("Bulgarian", "translated_bg"))
graph_builder.add_node("translate_it", create_translate_node("Italian", "translated_it"))

graph_builder.add_edge(START, "research")
graph_builder.add_edge("research", "write")
graph_builder.add_edge("write", "translate_bg")
graph_builder.add_edge("write", "translate_it")
graph_builder.add_edge("translate_bg", END)
graph_builder.add_edge("translate_it", END)

graph = graph_builder.compile()

In [ ]:
display_graph(graph, Path("/content/graph.png"))

In [ ]:
final_state = graph.invoke(input={ "topic": "What is the application of multi-agent systems in 2026?" })

In [ ]:
final_state